In [1]:
import numpy as np
import pandas as pd
import scipy.sparse as sparse
import itertools
import matplotlib
import matplotlib.patches as mpatches
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm
from datetime import date
import re
# from tqdm import tqdm
import time
# from pybedtools import BedTool
import random
import math
import pickle
import seaborn as sns
from scipy.stats import percentileofscore, pearsonr, rankdata, describe, mode, fisher_exact
# from scipy.spatial.distance import jaccard, euclidean, cosine
# import functools
# import operator
import gzip
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.metrics import pairwise_distances
import sys
import os
from functools import partial
from contextlib import closing
from multiprocessing import Pool
import re
import h5py
import tabix
import upsetplot
from wordcloud import WordCloud
from sklearn.decomposition import NMF, non_negative_factorization
from collections import Counter
from scipy.stats import hypergeom
from scipy.stats.mstats import gmean
from scipy.cluster import hierarchy

today = date.today()

In [2]:
def kde_wrapper(data, bw=0.1):
    fig = plt.figure()
    fig.patch.set_facecolor('white')
    
    sns.kdeplot(data, bw_adjust=bw)
    print(describe(data))
    print(mode(data))
    return fig

**Gencode file with expanded regulatory domains (as in GREAT)**

In [3]:
f = open('/home/ctrader/region_similarity/data/mart_export_protein_coding_genes_unique_with_proteinID_strand_CT20220919.txt', 'r')
data = []
colnames = f.readline().strip().split('\t')
colnames.append('TSS start (bp)')
colnames.append('TSS end (bp)')
colnames.append('Basal start (bp)')
colnames.append('Basal end (bp)')
colnames.append('Domain start (bp)')
colnames.append('Domain end (bp)')
data.append(colnames)
for line in f.readlines():
    line = line.strip().split('\t')
    #getting the TSS and basal locations
    if int(line[6]) == -1:
        line.append(str(int(line[2])-1))
        line.append(line[2])
        line.append(str(int(line[2])-1000))
        line.append(str(int(line[2])+5000))
        line.append(str(int(line[1])-1000000))
        line.append(str(int(line[2])+1000000))
    elif int(line[6]) == 1:
        line.append(line[1])
        line.append(str(int(line[1])+1))
        line.append(str(int(line[1])-5000))
        line.append(str(int(line[1])+1000))
        line.append(str(int(line[1])-1000000))
        line.append(str(int(line[2])+1000000))
    data.append(line)
f.close()

data = np.array(data)

In [4]:
#limiting domain to just the distance to another genes basal region
chrm = data[1:, 0]
tss = data[1:, 7].astype(int)
domain_m = data[1:, 9:].astype(int)
names = data[1:, 3]

domain_m = np.where(domain_m < 1, 1, domain_m)

In [5]:
t = time.time()
for gene in range(len(domain_m)):
    if gene%10000==0:
        print(gene)
    
    gene_chrm = chrm[gene]
    gene_tss = tss[gene]
    gene_name = names[gene]
    gene_dom = domain_m[gene, :]
    
    #filtering out wrong chromosomes
    chrm_mask = chrm==gene_chrm
    chrm_doms = domain_m[chrm_mask, :]
    chrm_tss = tss[chrm_mask]
    chrm_names = names[chrm_mask]
    
    #filtering out other annotations of the same gene
    name_mask = chrm_names!=gene_name
    chrm_name_doms = chrm_doms[name_mask, :]
    chrm_name_tss = chrm_tss[name_mask]
    
    #find all the genes to the left and right
    left_genes = chrm_name_tss<gene_tss
    right_genes = chrm_name_tss>gene_tss
    left_doms = chrm_name_doms[left_genes, :]
    right_doms = chrm_name_doms[right_genes, :]
    #if there are any genes, find the closest basal region
    if len(left_doms) > 0:
        max_left_basal_end = np.max(left_doms[:, 1])
    else:
        max_left_basal_end = gene_dom[2]
    if len(right_doms) > 0:
        min_right_basal_start = np.min(right_doms[:, 0])
    else:
        min_right_basal_start = gene_dom[3]
    #setting the domain for each gene
    gene_dom[2] = np.min([np.max([max_left_basal_end, gene_dom[2]]), gene_dom[0]])
    gene_dom[3] = np.max([np.min([min_right_basal_start, gene_dom[3]]), gene_dom[1]])

0
10000
20000
30000
40000
50000
60000
70000
80000


In [6]:
#changing domain values
data[1:, 9:] = domain_m.astype(str)

In [ ]:
outfile = '/home/ctrader/region_similarity/data/mart_export_protein_coding_genes_unique_with_proteinID_strand_CT20220920.extendeddomain.txt'
np.savetxt(outfile, data, delimiter='\t', fmt='%s')